Constants

In [ ]:
INPUT_ALPHA = set(">}])x!<{[(")
STACK_ALPHA = set("!<{[(")
EMPTY_STACK_SYMBOL = "Z"
EPSILON_SYMBOL = "E"
FINAL_STATES = set([2])
START_STATE = 0
BRCKT_CLOSE = {open : close for open, close in  ["<>", "{}", "[]", "()"]}

In [ ]:
class Transition:
    def __init__(self, symbol: str, top: str, replaced_top: str, to_state: int):
        self.symbol = symbol
        self.top = top
        self.replaced_top = replaced_top
        self.to_state = to_state
    
    def __repr__(self):
        return f"({self.symbol}, {self.top}, {self.replaced_top})"

class State:
    input_symbols = set()
    def __init__(self, symbol: str, transitions: dict[tuple, Transition]):
        self.symbol = symbol
        self.transitions = transitions
    
    def is_valid_transition(self, input_c, top_c):
        return (input_c, top_c) in self.transitions

    def get_transition(self, input_c, top_c):
        return self.transitions[(input_c, top_c)]


In [ ]:
states = [
    State(
        symbol="q0",
        transitions={
            ("!", EMPTY_STACK_SYMBOL) : Transition("!", EMPTY_STACK_SYMBOL, "!"+EMPTY_STACK_SYMBOL, 1)
        }
    ),
    State(
        symbol="q1",
        transitions={
            **{("x", top) : Transition("x", top, top, 1) for top in INPUT_ALPHA},
            **{(open_b, top) : Transition(open_b, top, open_b+top, 1) for open_b in BRCKT_CLOSE.keys() for top in INPUT_ALPHA},
            **{(close, open) : Transition(close, open, "", 1) for open, close in BRCKT_CLOSE.items()},
            **{("!", "!") : Transition('!', "!", "", 2)}
        }
    ),
    State(
        symbol="q2",
        transitions={(EPSILON_SYMBOL, EMPTY_STACK_SYMBOL) : Transition(EPSILON_SYMBOL, EMPTY_STACK_SYMBOL, EMPTY_STACK_SYMBOL, 2)}
    )
]


In [ ]:
def is_balanced(input_string):
    print(f"Processing {input_string}")

    stack = [EMPTY_STACK_SYMBOL]
    s_i = START_STATE
    i = 0
    while i <= len(input_string):
        input_c = (input_string[i] if i < len(input_string) else "E")
        top_c = stack[-1]

        print(f"ID: ({states[s_i].symbol}, {input_string[i:] if i < len(input_string) else "E"}, {''.join(reversed(stack))})")
        if not states[s_i].is_valid_transition(input_c, top_c):
            break
            
        transition = states[s_i].get_transition(input_c, top_c)

        if transition.replaced_top == "":
            stack.pop()
        else:
            stack[-1] = transition.replaced_top[-1]
            for c in reversed(transition.replaced_top[:-1]):
                stack.append(c)
        
        s_i = transition.to_state
        i += 1

    if s_i in FINAL_STATES and i > len(input_string):
        print(f"{states[s_i].symbol} is a final state.")
        print(f"{input_string} is valid and has balanced brackets.")
        return True
    elif s_i not in FINAL_STATES and i > len(input_string):
        print(f"Invalid string. {states[s_i].symbol} is not a final state.")
    else:
        print(f"Invalid string. Failed at position {i+1}.")
        print(f"Remaining unprocessed input string: {input_string[i:]}")
    return False
        


In [36]:
def main1():
    file = open("input.txt")
    input_strings = [s.strip() for s in file.readlines()]
    
    for s in input_strings:
        is_balanced(s)
        print() # to separate outputs of each input string

main1()

Processing !xx[x({xx})[xxx]x]<xxx>x!
ID: (q0, !xx[x({xx})[xxx]x]<xxx>x!, Z)
ID: (q1, xx[x({xx})[xxx]x]<xxx>x!, !Z)
ID: (q1, x[x({xx})[xxx]x]<xxx>x!, !Z)
ID: (q1, [x({xx})[xxx]x]<xxx>x!, !Z)
ID: (q1, x({xx})[xxx]x]<xxx>x!, [!Z)
ID: (q1, ({xx})[xxx]x]<xxx>x!, [!Z)
ID: (q1, {xx})[xxx]x]<xxx>x!, ([!Z)
ID: (q1, xx})[xxx]x]<xxx>x!, {([!Z)
ID: (q1, x})[xxx]x]<xxx>x!, {([!Z)
ID: (q1, })[xxx]x]<xxx>x!, {([!Z)
ID: (q1, )[xxx]x]<xxx>x!, ([!Z)
ID: (q1, [xxx]x]<xxx>x!, [!Z)
ID: (q1, xxx]x]<xxx>x!, [[!Z)
ID: (q1, xx]x]<xxx>x!, [[!Z)
ID: (q1, x]x]<xxx>x!, [[!Z)
ID: (q1, ]x]<xxx>x!, [[!Z)
ID: (q1, x]<xxx>x!, [!Z)
ID: (q1, ]<xxx>x!, [!Z)
ID: (q1, <xxx>x!, !Z)
ID: (q1, xxx>x!, <!Z)
ID: (q1, xx>x!, <!Z)
ID: (q1, x>x!, <!Z)
ID: (q1, >x!, <!Z)
ID: (q1, x!, !Z)
ID: (q1, !, !Z)
ID: (q2, E, Z)
q2 is a final state.
!xx[x({xx})[xxx]x]<xxx>x! is valid and has balanced brackets.

Processing ![<]>!
ID: (q0, ![<]>!, Z)
ID: (q1, [<]>!, !Z)
ID: (q1, <]>!, [!Z)
ID: (q1, ]>!, <[!Z)
Invalid string. Failed at position 4.

In [37]:
def evaluate(input_string):
    def eval(start_i, open_b):
        x_count = 0
        i = start_i
        while i < len(input_string):
            if input_string[i] == "x":
                x_count += 1
                i += 1
            elif input_string[i] in "<{[(":
                xs, ends_at = eval(i+1, input_string[i])
                x_count += xs
                i = ends_at+1
            elif input_string[i] == '!' or input_string[i] == BRCKT_CLOSE[open_b]:
                break
        
        if open_b == "<":
            x_count *= 2
        elif open_b == "{":
            x_count += 1
        elif open_b == "[":
            x_count = 0
        elif open_b == "(":
            x_count = max(0, x_count-1)

        return (x_count, i)

    x_count, ends_at = eval(1, "!")
    assert(ends_at == len(input_string)-1)
    return x_count

In [38]:
def main2():
    file = open("input.txt")
    input_strings = [s.strip() for s in file.readlines()]
    
    for s in input_strings:
        if is_balanced(s):
            print(f"{s} - Resulting number of x's: {evaluate(s)}")
        else:
            print(f"{s} - Invalid string.")

main2()

Processing !xx[x({xx})[xxx]x]<xxx>x!
ID: (q0, !xx[x({xx})[xxx]x]<xxx>x!, Z)
ID: (q1, xx[x({xx})[xxx]x]<xxx>x!, !Z)
ID: (q1, x[x({xx})[xxx]x]<xxx>x!, !Z)
ID: (q1, [x({xx})[xxx]x]<xxx>x!, !Z)
ID: (q1, x({xx})[xxx]x]<xxx>x!, [!Z)
ID: (q1, ({xx})[xxx]x]<xxx>x!, [!Z)
ID: (q1, {xx})[xxx]x]<xxx>x!, ([!Z)
ID: (q1, xx})[xxx]x]<xxx>x!, {([!Z)
ID: (q1, x})[xxx]x]<xxx>x!, {([!Z)
ID: (q1, })[xxx]x]<xxx>x!, {([!Z)
ID: (q1, )[xxx]x]<xxx>x!, ([!Z)
ID: (q1, [xxx]x]<xxx>x!, [!Z)
ID: (q1, xxx]x]<xxx>x!, [[!Z)
ID: (q1, xx]x]<xxx>x!, [[!Z)
ID: (q1, x]x]<xxx>x!, [[!Z)
ID: (q1, ]x]<xxx>x!, [[!Z)
ID: (q1, x]<xxx>x!, [!Z)
ID: (q1, ]<xxx>x!, [!Z)
ID: (q1, <xxx>x!, !Z)
ID: (q1, xxx>x!, <!Z)
ID: (q1, xx>x!, <!Z)
ID: (q1, x>x!, <!Z)
ID: (q1, >x!, <!Z)
ID: (q1, x!, !Z)
ID: (q1, !, !Z)
ID: (q2, E, Z)
q2 is a final state.
!xx[x({xx})[xxx]x]<xxx>x! is valid and has balanced brackets.
!xx[x({xx})[xxx]x]<xxx>x! - Resulting number of x's: 9
Processing ![<]>!
ID: (q0, ![<]>!, Z)
ID: (q1, [<]>!, !Z)
ID: (q1, <]>!, [!Z)
ID:

In [28]:
# testing {states} content
for i in range(3):
    print(f"State {i}")
    for k,t in states[i].transitions.items():
        print(k, t)
        assert(k == (t.symbol, t.top))
    print()

State 0
('!', 'Z') (!, Z, !Z)

State 1
('x', '>') (x, >, >)
('x', '[') (x, [, [)
('x', 'x') (x, x, x)
('x', '<') (x, <, <)
('x', '!') (x, !, !)
('x', '}') (x, }, })
('x', ')') (x, ), ))
('x', '(') (x, (, ()
('x', ']') (x, ], ])
('x', '{') (x, {, {)
('<', '>') (<, >, <>)
('<', '[') (<, [, <[)
('<', 'x') (<, x, <x)
('<', '<') (<, <, <<)
('<', '!') (<, !, <!)
('<', '}') (<, }, <})
('<', ')') (<, ), <))
('<', '(') (<, (, <()
('<', ']') (<, ], <])
('<', '{') (<, {, <{)
('{', '>') ({, >, {>)
('{', '[') ({, [, {[)
('{', 'x') ({, x, {x)
('{', '<') ({, <, {<)
('{', '!') ({, !, {!)
('{', '}') ({, }, {})
('{', ')') ({, ), {))
('{', '(') ({, (, {()
('{', ']') ({, ], {])
('{', '{') ({, {, {{)
('[', '>') ([, >, [>)
('[', '[') ([, [, [[)
('[', 'x') ([, x, [x)
('[', '<') ([, <, [<)
('[', '!') ([, !, [!)
('[', '}') ([, }, [})
('[', ')') ([, ), [))
('[', '(') ([, (, [()
('[', ']') ([, ], [])
('[', '{') ([, {, [{)
('(', '>') ((, >, (>)
('(', '[') ((, [, ([)
('(', 'x') ((, x, (x)
('(', '<') ((, <, (<)
('(